In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('diabetes.csv')
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [3]:
X = df.drop('Outcome',axis=1)
Y = df['Outcome']

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scale = scaler.fit_transform(X)



In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X_scale,Y,test_size=0.2,random_state=42)

In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import kerastuner as kt


/var/folders/83/vkcpj0h51_1gwqn9b___8hh00000gn/T/ipykernel_72484/1517588427.py:3: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [15]:
def build_model(hp):
    model = Sequential()

    units = hp.Int('units', min_value=8, max_value=128)
    model.add(Dense(units=units,activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='rmsprop',loss = 'binary_crossentropy',metrics=['accuracy'])
    return model



In [16]:
tuner = kt.RandomSearch(
    build_model,
    objective = 'val_accuracy',
    max_trials = 5,
    directory = 'my_dir',
    project_name = 'name01'
)

In [17]:
tuner.search(X_train,Y_train,epochs=5,validation_data=(X_test,Y_test))

Trial 5 Complete [00h 00m 01s]
val_accuracy: 0.7467532753944397

Best val_accuracy So Far: 0.7597402334213257
Total elapsed time: 00h 00m 04s


In [18]:
tuner.results_summary()

Results summary
Results in my_dir/name01
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 1 summary
Hyperparameters:
units: 111
Score: 0.7597402334213257

Trial 2 summary
Hyperparameters:
units: 19
Score: 0.7597402334213257

Trial 0 summary
Hyperparameters:
units: 32
Score: 0.7532467246055603

Trial 4 summary
Hyperparameters:
units: 36
Score: 0.7467532753944397

Trial 3 summary
Hyperparameters:
units: 34
Score: 0.7142857313156128


In [19]:
tuner.get_best_hyperparameters()[0].values

{'units': 111}

In [20]:
model = tuner.get_best_models(num_models=1)[0]

In [21]:
model.fit(X_train,Y_train,epochs=100,validation_data=(X_test,Y_test),initial_epoch=6)

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7655 - loss: 0.4840 - val_accuracy: 0.7597 - val_loss: 0.5093
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7704 - loss: 0.4697 - val_accuracy: 0.7597 - val_loss: 0.5077
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7818 - loss: 0.4617 - val_accuracy: 0.7597 - val_loss: 0.5049
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7769 - loss: 0.4561 - val_accuracy: 0.7662 - val_loss: 0.5028
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7801 - loss: 0.4506 - val_accuracy: 0.7662 - val_loss: 0.5045
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7915 - loss: 0.4473 - val_accuracy: 0.7597 - val_loss: 0.5074
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7785 - loss: 0.4440 - val_accuracy: 0.7662 - val_loss: 0.5082
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7801 - loss: 0.4413 - val_accuracy: 0.766